# Building hybrid search on Couchbase

*Vector, then keyword, then both in one request, then filtered — each step a working search.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/retrieval/01_building_hybrid_search.ipynb)
[![Open in GitHub Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** One Couchbase Search index can carry text, vectors and filterable fields, and answer all three in a single request.
**Result.** Four working searches over 6,482 real products — and two queries that pick opposite winners, which is why there is a second notebook.
**Requires.** couchbase · local-embeddings
**Read** ~10 min · **Run** ~4 min · **Cost** $0.00 — no LLM calls

Search on a product catalogue is a good place to learn this, because the two obvious approaches
fail in opposite directions and you can see it happen.

**Keyword search** matches the words you typed. Ask for *nautical platters* and it will happily
return a nautical headboard, because "nautical" is right there and it has no idea that
"platters" was the important word.

**Vector search** matches meaning. Ask for a *king poster bed* and it may offer you a queen one,
because in embedding space those are nearly the same thing — and to a shopper they are not.

The fix is to use both, and Couchbase will do that in **one request against one index**. This
notebook builds that up a step at a time: vector alone, keyword alone, both, then both with a
filter. Every step runs.

What it deliberately does *not* do is tell you which to ship. By the end you will have two
queries that disagree about the winner, and no way to settle it from three results apiece.
Settling it is [`retrieval/02`](02_which_parts_helped.ipynb).

In [ ]:
# Setup: load the helpers, install anything missing, check what this notebook needs.
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "local-embeddings"])

## 1. One document per product

These are real Wayfair listings, with the department they were filed under. Four fields matter,
and they are going to be indexed three different ways:

| field | indexed as | why |
| --- | --- | --- |
| `text` | analysed text | so keyword search can match individual words in it |
| `embedding` | vector | so semantic search can find it by meaning |
| `department` | keyword | so it can be *filtered* on, exactly, not searched |
| `category_path` | keyword | same, for narrower filters |

The distinction in the last two is the one that catches people out, and section 3 comes back
to it.

In [2]:
# Load the WANDS products and build the short text each one is indexed by.
from cbnb.datasets import load_wands_eval_set

catalogue = load_wands_eval_set().products.copy()
catalogue["category_hierarchy"] = catalogue.category_hierarchy.fillna("")
catalogue["department"] = catalogue.category_hierarchy.str.split(" / ").str[0]
catalogue["text"] = (catalogue.product_name.fillna("") + " "
                     + catalogue.product_class.fillna("")).str.strip()

print(f"{len(catalogue):,} products across {catalogue.department.nunique()} departments")
catalogue[["product_id", "text", "department"]].head(4)

6,482 products across 33 departments


,product_id,text,department
0,4,baldwin prestige alcott passage knob with roun...,Home Improvement
1,13,king tufted low profile standard bed Beds,Furniture
2,30,lugent 70.87 '' h x 70.87 '' w x 11.8 '' d ste...,Furniture
3,33,tatianna 2 piece pine solid wood floating shel...,Storage & Organization


In [3]:
# Embed each product and store it in Couchbase.
from cbnb.couchbase_io import connect, ensure_collection, upsert_docs
from cbnb.embeddings import Embedder

BUCKET, SCOPE, COLLECTION = settings.cb_bucket, "hybrid_retrieval", "products"
INDEX = "products_hybrid"

embedder = Embedder()                       # all-MiniLM-L6-v2, on this machine
vectors = embedder.encode(catalogue.text.tolist())
print(f"{vectors.shape[0]:,} embeddings of {vectors.shape[1]} dimensions")

cluster = connect(settings)
collection = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)
upsert_docs(collection, {
    str(row.product_id): {
        "type": COLLECTION,
        "product_id": int(row.product_id),
        "text": row.text,
        "product_name": row.product_name,
        "department": row.department,
        "category_path": row.category_hierarchy,
        "embedding": vector.tolist(),
    }
    for row, vector in zip(catalogue.itertuples(index=False), vectors)
})

6,482 embeddings of 384 dimensions


  upserted 200/6482

  upserted 400/6482

  upserted 600/6482

  upserted 800/6482

...

  upserted 6000/6482

  upserted 6200/6482

  upserted 6400/6482

  upserted 6482/6482

6482

## 2. One index for all of it

`ensure_vector_index` writes a scope-level Search index. The three field lists map onto the
three jobs above — and note that the vector, the text and the filters all live in the *same*
index. That is what makes a single-request hybrid query possible later.

<details>
<summary>What this generates, for the curious</summary>

Each field becomes an entry in the index's type mapping. The vector field carries its
dimensions and similarity metric:

```json
"embedding": {"fields": [{"name": "embedding", "type": "vector",
                          "dims": 384, "similarity": "dot_product",
                          "vector_index_optimized_for": "recall"}]}
```

A text field gets a language analyzer, which lowercases and stems, so *tables* matches
*table*:

```json
"text": {"fields": [{"name": "text", "type": "text", "analyzer": "en",
                     "include_term_vectors": true, "store": true}]}
```

A keyword field gets the `keyword` analyzer, which does none of that — see section 3.

```json
"department": {"fields": [{"name": "department", "type": "text",
                           "analyzer": "keyword", "docvalues": true}]}
```

`cbnb/couchbase_io.py` has the full definition, including the `scorch` store settings and the
`scope.collection.type_field` document mapping that scopes the index to this collection.
</details>

Building it takes a minute or two; `wait_for_index` blocks until the index can actually answer
for every document, which is not the same as the index existing.

In [4]:
# Build one Search index for text, vectors and filters; wait until it's ready.
from cbnb.couchbase_io import ensure_vector_index, wait_for_index

ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name=COLLECTION,
    index_name=INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["text"],                              # analysed -> searchable words
    keyword_fields=["department", "category_path"],    # verbatim -> filterable values
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(catalogue))

  indexed 6482/6482 (ready)   

  indexed 6482/6482 (ready)   

6482

## 3. Vector search on its own

Embed the query with the same model that embedded the products, then ask for its nearest
neighbours.

`num_candidates` is the knob worth knowing: the index examines that many candidates before
handing back the best `k`. Raise it for better recall, pay for it in latency.

In [5]:
# Vector search: find products whose meaning is closest to each query.
import pandas as pd

from cbnb.couchbase_io import vector_search

pd.set_option("display.max_colwidth", None)   # show whole product names, not "Servi..."

WHERE = dict(bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX)

# Two queries used throughout: each is one approach's strength and the other's weakness.
NOUN_MATTERS = "nautical platters"
WORDS_MATTER = "king poster bed"


def side_by_side(searches, title=""):
    """Search results as a table: one column per search, one row per rank."""
    table = pd.DataFrame({
        label: pd.Series([hit["text"] for hit in hits], index=range(1, len(hits) + 1))
        for label, hits in searches.items()
    }).fillna("")
    table.columns.name = title   # shown in the corner; the row labels are ranks
    return table


def by_vector(query, k=5):
    return vector_search(cluster, **WHERE, vector_field="embedding",
                         query_vector=embedder.encode_one(query),
                         k=k, num_candidates=200, fields=["text"])


side_by_side({NOUN_MATTERS: by_vector(NOUN_MATTERS),
              WORDS_MATTER: by_vector(WORDS_MATTER)}, title="vector")

vector,nautical platters,king poster bed
1,boat shaped platter Serving Dishes & Platters,essential king four poster bed Beds
2,coastal platter Serving Dishes & Platters,kingstown four poster bed Beds
3,coastal medium platter Serving Dishes & Platters,ameed queen four poster bed
4,seaside platter Serving Dishes & Platters,amed queen four poster bed
5,high seas knot rope platter Serving Dishes & Platters,king solid wood four poster bed Beds


Look at what it did with each.

For *nautical platters* it understood that **platters** was the point, and returned platters —
boat-shaped and coastal ones. None of them contain the word "nautical" at all. That is the
thing keyword search cannot do.

For *king poster bed* it returned four-poster beds, and then offered a **queen** one. In
embedding space *king* and *queen* are close together. To someone buying a bed they are not
interchangeable at all, and no amount of tuning the vector search fixes that — the information
is being blurred on purpose by the model.

## 4. Keyword search on its own

A `MatchQuery` against the analysed `text` field: BM25, the thing search engines did for
twenty-five years before embeddings, and still the right tool for exact words.

In [6]:
# Keyword search: find products that share the query's words.
import couchbase.search as search

from cbnb.couchbase_io import text_search


def by_keyword(query, k=5):
    return text_search(cluster, **WHERE,
                       text_query=search.MatchQuery(query, field="text"),
                       k=k, fields=["text"])


side_by_side({NOUN_MATTERS: by_keyword(NOUN_MATTERS),
              WORDS_MATTER: by_keyword(WORDS_MATTER)}, title="keyword")

keyword,nautical platters,king poster bed
1,wilton armetale nautical long bread tray Serving Dishes & Platters,essential king four poster bed Beds
2,nautical upholstered panel headboard Headboards,king solid wood four poster bed Beds
3,nautical anchor wall décor Wall Décor,distressed wood king four poster bed Beds
4,nautical and coastal blue rustic anchor nautical watercrafts - graphic art print Wall Art,aszia king low profile four poster bed Beds
5,nautical wood anchor wall décor Wall Décor,miraflores king solid wood low profile four poster bed Beds


Exactly the opposite failure, which is the good news.

*king poster bed* is handled precisely — every hit is a king four-poster, because "king" is a
word in the document and BM25 does not think "king" is a bit like "queen".

*nautical platters* goes wrong in a way that looks silly and is completely logical: a nautical
**headboard** and nautical **wall décor**. "Nautical" is a rare word, so BM25 weights it
heavily; "platters" is comparatively common. It has no concept that one is the product and the
other is a style.

### Analysed or keyword: the choice that bites

Both `text` and `department` are stored as `type: "text"` in the index. The analyzer is what
makes them behave completely differently:

- **`analyzer: "en"`** on `text` splits the value into words, lowercases them, and stems them.
  `"Coffee Tables & End Tables"` becomes searchable as `coffee`, `tabl`, `end`. Good for
  matching *part* of a value.
- **`analyzer: "keyword"`** on `department` keeps the entire value as **one term**.
  `"Décor & Pillows"` is a single token, matched only by exactly that string — accent,
  ampersand and all.

Use the wrong one and the symptom is confusing: filters that match nothing, or a filter on
"Furniture" that also matches "Outdoor Furniture". Filterable values want `keyword`; prose
wants an analyzer.

## 5. Both at once, in one request

Here is the part that is specifically a Couchbase capability rather than a technique: the
lexical query and the vector query go to the **same index in a single request**, and the
service combines the scores.

```python
SearchRequest.create(text_query).with_vector_search(VectorSearch.from_vector_query(...))
```

No second system to keep in sync, no fan-out to two engines, no fusion code of your own to
write and get subtly wrong. One round trip.

In [7]:
# Hybrid search: run both in one request and combine the scores.
from cbnb.couchbase_io import hybrid_search


def by_hybrid(query, k=5, prefilter=None):
    return hybrid_search(cluster, **WHERE, vector_field="embedding",
                         query_vector=embedder.encode_one(query),
                         text_query=search.MatchQuery(query, field="text"),
                         k=k, num_candidates=200, fields=["text"], prefilter=prefilter)


side_by_side({NOUN_MATTERS: by_hybrid(NOUN_MATTERS),
              WORDS_MATTER: by_hybrid(WORDS_MATTER)}, title="hybrid")

hybrid,nautical platters,king poster bed
1,wilton armetale nautical long bread tray Serving Dishes & Platters,essential king four poster bed Beds
2,coastal platter Serving Dishes & Platters,king solid wood four poster bed Beds
3,seaside platter Serving Dishes & Platters,distressed wood king four poster bed Beds
4,bay platter Serving Dishes & Platters,aszia king low profile four poster bed Beds
5,boat shaped platter Serving Dishes & Platters,miraflores king solid wood low profile four poster bed Beds


## 6. Narrowing it with a filter

If you already know the shopper is in one department — from a navigation click, a previous
query, or a classifier like the one in
[`enrich/01`](../enrich/01_hypothetical_classification.ipynb) — you can restrict the search to
it.

**Use a prefilter, not a post-filter.** A `prefilter` is applied *before* the nearest-neighbour
search, so the candidates are drawn from inside the filtered set and you still get `k` results.
Fetching `k` results and then discarding the ones in the wrong department is the version
everyone writes first, and it can leave you with two hits out of ten — or none.

This is where the `keyword` analyzer earns itself: `TermQuery` matches the department value
exactly, with no stemming to blur it.

In [8]:
# Hybrid search again, filtered to the Kitchen & Tabletop department.
departments = catalogue.department.value_counts().head(5)
print("largest departments:", ", ".join(departments.index[:3]), "...\n")

in_kitchen = search.TermQuery("Kitchen & Tabletop", field="department")
side_by_side({NOUN_MATTERS: by_hybrid(NOUN_MATTERS, prefilter=in_kitchen)},
             title="hybrid + Kitchen & Tabletop filter")

largest departments: Furniture, Décor & Pillows, Home Improvement ...



hybrid + Kitchen & Tabletop filter,nautical platters
1,wilton armetale nautical long bread tray Serving Dishes & Platters
2,seaside platter Serving Dishes & Platters
3,bay platter Serving Dishes & Platters
4,fish platter Serving Dishes & Platters
5,drift platter Serving Dishes & Platters


## 7. Four searches, all of them plausible

Put the same query through everything built so far, side by side.

In [9]:
# Compare every approach side by side on both queries.
from IPython.display import display

for query, where in [(NOUN_MATTERS, in_kitchen),
                     (WORDS_MATTER, search.TermQuery("Furniture", field="department"))]:
    display(side_by_side({
        "keyword only": by_keyword(query, k=3),
        "vector only": by_vector(query, k=3),
        "hybrid": by_hybrid(query, k=3),
        "hybrid + filter": by_hybrid(query, k=3, prefilter=where),
    }, title=repr(query)))

'nautical platters',keyword only,vector only,hybrid,hybrid + filter
1,wilton armetale nautical long bread tray Serving Dishes & Platters,boat shaped platter Serving Dishes & Platters,wilton armetale nautical long bread tray Serving Dishes & Platters,wilton armetale nautical long bread tray Serving Dishes & Platters
2,nautical upholstered panel headboard Headboards,coastal platter Serving Dishes & Platters,coastal platter Serving Dishes & Platters,seaside platter Serving Dishes & Platters
3,nautical anchor wall décor Wall Décor,coastal medium platter Serving Dishes & Platters,seaside platter Serving Dishes & Platters,bay platter Serving Dishes & Platters


'king poster bed',keyword only,vector only,hybrid,hybrid + filter
1,essential king four poster bed Beds,essential king four poster bed Beds,essential king four poster bed Beds,essential king four poster bed Beds
2,king solid wood four poster bed Beds,kingstown four poster bed Beds,king solid wood four poster bed Beds,king solid wood four poster bed Beds
3,distressed wood king four poster bed Beds,ameed queen four poster bed,distressed wood king four poster bed Beds,distressed wood king four poster bed Beds


Now the honest part, and it is not the one you might expect.

You *can* see a difference here — and that is exactly the trap.

On **`nautical platters`**, keyword-only is obviously the worst of the four: it leads with a
bread tray and, left unfiltered, wanders off to headboards. Vector-only looks best.

On **`king poster bed`**, the verdict reverses completely. Keyword-only is precise, every hit a
king four-poster, while vector-only offers a *queen* bed in third place. And three of the four
approaches return **identical** lists, so for that query the vector half contributed nothing at
all that you can see.

Two queries, two opposite winners, and one case where the clever thing made no difference. Pick
either query as your demo and you would walk away with a confident, well-evidenced, wrong
conclusion about which approach to ship.

That is the whole problem. Eyeballing results does not scale to a decision, because the thing
you need to know is not "is this list good" but "**is this list better, on average, across the
queries I will actually get**" — and no amount of staring at three hits answers that.

[`retrieval/02`](02_which_parts_helped.ipynb) answers it. It reuses the index you just built,
and 480 real queries that Wayfair paid humans to judge.

## Where to take this

- **Tune `num_candidates`.** 200 was a round number. It trades latency for recall, and `02`
  shows you how to see the effect instead of guessing at it.
- **Filter on more than department.** `category_path` is indexed the same way; a `PrefixQuery`
  on it narrows to a branch of the taxonomy rather than a whole department.
- **Weight the two halves.** This notebook takes whatever balance the Search service defaults
  to. Production systems tune it — and tuning without measuring is how you make things worse
  confidently.

> **On Couchbase AI Data Plane** — the embedding step here runs on your machine, and the vectors
> go stale the moment a product description changes. The managed service generates and maintains
> them as documents change. The index itself — one definition carrying text, vectors and
> filterable fields, answering all three in one request — is the same either way.

In [10]:
# Optional clean-up: delete what this notebook created. Uncomment to run.
# `retrieval/02` and `03` reuse it, so leave it in place if you plan to run them next.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)